# AI Gateway — Tenant Quota Management

Interactive notebook for managing, validating, and visualising per-tenant token budgets enforced by the APIM Redis-backed quota policy.

| Section | What it does |
|---------|-------------|
| **1 · Overview** | Load current settings from Terraform and display all tenants in a table + chart |
| **2 · Adjust Quota** | Change TPM / quota / period for any tenant, preview the budget impact, and push to APIM in one click |
| **3 · Compliance Test** | Fire real requests through the gateway and prove the quota is respected — with a table, bar chart, burn-down line, and budget gauge |

> **Prerequisites** — run from the `gen-ai-gateway-lite` folder with `az login` active and Terraform state available.


In [6]:
# -- Install packages (idempotent) --------------------------------------------
import subprocess, sys

_required = ["pandas", "plotly", "ipywidgets", "nbformat"]
for _p in _required:
    subprocess.run([sys.executable, "-m", "pip", "install", _p, "-q"], check=False)

# -- Imports ------------------------------------------------------------------
import json, re, ssl, urllib.request, urllib.error
from datetime import datetime, timezone, timedelta
from pathlib import Path
import math

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

print("All packages loaded.")


All packages loaded.


In [8]:
# -- Project root -------------------------------------------------------------
PROJECT_DIR = Path.cwd()
TFVARS_PATH = PROJECT_DIR / "terraform.tfvars"

assert TFVARS_PATH.exists(), (
    f"terraform.tfvars not found at {TFVARS_PATH}\n"
    "Open a terminal, cd into gen-ai-gateway-lite/, then restart this kernel."
)
print(f"Project root : {PROJECT_DIR}")
print(f"tfvars       : {TFVARS_PATH}")

# -- Period -> minutes lookup -------------------------------------------------
_PERIOD_MIN = {
    "Hourly":  60,
    "Daily":   1_440,
    "Weekly":  10_080,
    "Monthly": 44_640,
    "Yearly":  525_960,
}

def budget_stats(tpm, quota, period):
    requests_approx  = math.ceil(quota / 800)
    full_tpm_minutes = round(quota / tpm, 1) if tpm else 0
    return requests_approx, full_tpm_minutes

# -- Terraform output reader --------------------------------------------------
def tf_output(name):
    r = subprocess.run(
        ["terraform", "output", "-json", name],
        capture_output=True, text=True, cwd=str(PROJECT_DIR),
    )
    if r.returncode != 0:
        raise RuntimeError(f"terraform output '{name}' failed:\n{r.stderr.strip()}")
    return json.loads(r.stdout)

# -- HTTP chat helper ---------------------------------------------------------
_ssl_ctx = ssl.create_default_context()

def chat(gateway_url, api_key, prompt, max_tokens=800):
    url  = (f"{gateway_url}/openai/deployments/gpt-4o-mini/chat/completions"
            "?api-version=2024-10-21")
    data = json.dumps({
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
    }).encode()
    req = urllib.request.Request(url, data=data, headers={
        "Content-Type": "application/json",
        "api-key": api_key,
    })
    try:
        with urllib.request.urlopen(req, context=_ssl_ctx, timeout=60) as resp:
            body = json.loads(resp.read())
            return resp.status, body.get("usage", {}), dict(resp.headers)
    except urllib.error.HTTPError as e:
        raw = e.read().decode()
        hdr = dict(e.headers) if hasattr(e, "headers") else {}
        try:    body = json.loads(raw)
        except: body = {"raw": raw}
        return e.code, body, hdr

# -- terraform.tfvars in-place patcher ----------------------------------------
def patch_tfvars(slug, tpm, quota, period):
    lines = TFVARS_PATH.read_text(encoding="utf-8").splitlines(keepends=True)
    in_block, depth, out = False, 0, []
    for line in lines:
        if re.match(rf"^\s*{re.escape(slug)}\s*=\s*{{", line):
            in_block, depth = True, 1
            out.append(line); continue
        if in_block:
            depth += line.count("{") - line.count("}")
            if depth <= 0:
                in_block = False; out.append(line); continue
            ind = re.match(r"^(\s*)", line).group(1)
            if   re.match(r"^\s*tokens_per_minute\s*=",  line): out.append(f"{ind}tokens_per_minute  = {tpm}\n")
            elif re.match(r"^\s*token_quota\s*=\s*\d",   line): out.append(f"{ind}token_quota        = {quota}\n")
            elif re.match(r"^\s*token_quota_period\s*=", line): out.append(f'{ind}token_quota_period = "{period}"\n')
            else: out.append(line)
        else:
            out.append(line)
    TFVARS_PATH.write_text("".join(out), encoding="utf-8")

print("Helpers ready.")


Project root : c:\Users\jmasengesho\Documents\developer\AI\AI-Landing-Zones\gen-ai-gateway-lite
tfvars       : c:\Users\jmasengesho\Documents\developer\AI\AI-Landing-Zones\gen-ai-gateway-lite\terraform.tfvars
Helpers ready.


In [9]:
# -- Load Terraform outputs ---------------------------------------------------
print("Loading Terraform outputs...")
gateway     = tf_output("apim_gateway_url")
tenant_keys = tf_output("apim_tenant_subscription_keys")
tenant_cfg  = tf_output("tenant_config")
print(f"  Gateway : {gateway}")
print(f"  Tenants : {list(tenant_cfg.keys())}")

# -- Build overview DataFrame -------------------------------------------------
rows = []
for slug, cfg in tenant_cfg.items():
    req_approx, full_tpm_min = budget_stats(
        cfg["tokens_per_minute"], cfg["token_quota"], cfg["token_quota_period"]
    )
    rows.append({
        "Display Name":          cfg["display_name"],
        "Slug":                  slug,
        "TPM":                   cfg["tokens_per_minute"],
        "Quota (tokens)":        cfg["token_quota"],
        "Period":                cfg["token_quota_period"],
        "Requests @ 800 tok":    req_approx,
        "Full-TPM budget (min)": full_tpm_min,
    })
df_overview = pd.DataFrame(rows)

def _style_budget(val):
    if isinstance(val, (int, float)):
        if val < 10:  return "background-color: #fde7e9; color: #d13438; font-weight:bold"
        if val < 60:  return "background-color: #fff4ce"
    return ""

display(HTML("<h3>Current Tenant Configuration</h3>"))
display(
    df_overview.style
    .format({
        "TPM":                   "{:,}",
        "Quota (tokens)":        "{:,}",
        "Requests @ 800 tok":    "~{:,}",
        "Full-TPM budget (min)": "{:.1f} min",
    })
    .map(_style_budget, subset=["Full-TPM budget (min)"])
    .set_table_styles([
        {"selector": "thead th",               "props": [("background-color","#0078d4"),("color","white"),("font-weight","bold"),("padding","9px 14px")]},
        {"selector": "td",                     "props": [("padding","7px 14px")]},
        {"selector": "tr:nth-of-type(even) td","props": [("background-color","#f0f7ff")]},
    ])
    .hide(axis="index")
)

fig_overview = go.Figure(data=[go.Bar(
    x=df_overview["Display Name"],
    y=df_overview["Quota (tokens)"],
    text=df_overview["Period"],
    textposition="outside",
    marker_color=["#0078d4","#107c10","#8764b8","#ca5010"],
)])
fig_overview.update_layout(
    title="Token Quota per Tenant",
    yaxis_title="Quota (tokens)",
    xaxis_title="",
    height=320, margin=dict(l=50, r=20, t=50, b=40),
    showlegend=False,
)
fig_overview.show()


Loading Terraform outputs...
  Gateway : https://apim-tf34lab09.azure-api.net
  Tenants : ['adventure-works', 'contoso', 'fabrikam', 'floor-works']


Display Name,Slug,TPM,Quota (tokens),Period,Requests @ 800 tok,Full-TPM budget (min)
Adventure Works,adventure-works,"10,000","3,000,000",Monthly,"~3,750",300.0 min
Contoso Corp,contoso,"1,000","500,000",Monthly,~625,500.0 min
Fabrikam Inc,fabrikam,"2,000","5,000",Hourly,~7,2.5 min
Floor Works,floor-works,"5,000","1,500,000",Monthly,"~1,875",300.0 min


## Adjust Tenant Quota

Use the controls below to change a tenant's **TPM cap**, **token quota**, or **reset period**.

1. Select a tenant — the current values populate automatically.
2. Edit the fields.
3. Click **Preview** to see a side-by-side comparison and a budget-change chart.
4. Click **Apply to APIM** to patch `terraform.tfvars` and run `terraform apply` (≈ 30 s).


In [10]:
# -- Widget definitions -------------------------------------------------------
_slug_opts = [(f"{v['display_name']}  ({k})", k) for k, v in tenant_cfg.items()]

slug_dd   = widgets.Dropdown(options=_slug_opts, description="Tenant:",
                             layout=widgets.Layout(width="360px"))
tpm_in    = widgets.BoundedIntText(min=100, max=100_000, step=500, description="TPM:",
                                   layout=widgets.Layout(width="230px"))
quota_in  = widgets.BoundedIntText(min=1_000, max=50_000_000, step=1_000, description="Quota:",
                                   layout=widgets.Layout(width="230px"))
period_dd = widgets.Dropdown(options=["Hourly","Daily","Weekly","Monthly","Yearly"],
                             description="Period:", layout=widgets.Layout(width="210px"))

preview_btn = widgets.Button(description=" Preview",       button_style="info",    icon="table",
                             layout=widgets.Layout(width="160px"))
apply_btn   = widgets.Button(description=" Apply to APIM", button_style="warning", icon="cloud-upload",
                             layout=widgets.Layout(width="180px"), disabled=True)
apply_out   = widgets.Output()

def _seed(change=None):
    cfg = tenant_cfg[slug_dd.value]
    tpm_in.value    = cfg["tokens_per_minute"]
    quota_in.value  = cfg["token_quota"]
    period_dd.value = cfg["token_quota_period"]
    apply_btn.disabled = True
    with apply_out: clear_output()

slug_dd.observe(_seed, names="value")
_seed()

def on_preview(b):
    slug = slug_dd.value
    old  = tenant_cfg[slug]
    old_req, old_min = budget_stats(old["tokens_per_minute"], old["token_quota"], old["token_quota_period"])
    new_req, new_min = budget_stats(tpm_in.value, quota_in.value, period_dd.value)

    comparison = pd.DataFrame([
        {"Setting": "TPM",                  "Current": f"{old['tokens_per_minute']:,}",  "New": f"{tpm_in.value:,}",    "Delta": f"{tpm_in.value - old['tokens_per_minute']:+,}"},
        {"Setting": "Quota (tokens)",       "Current": f"{old['token_quota']:,}",         "New": f"{quota_in.value:,}", "Delta": f"{quota_in.value - old['token_quota']:+,}"},
        {"Setting": "Reset period",         "Current": old["token_quota_period"],          "New": period_dd.value,       "Delta": ""},
        {"Setting": "Requests @ 800 tok",   "Current": f"~{old_req:,}",                   "New": f"~{new_req:,}",       "Delta": f"{new_req - old_req:+,}"},
        {"Setting": "Full-TPM budget (min)","Current": f"{old_min} min",                  "New": f"{new_min} min",      "Delta": f"{new_min - old_min:+.1f} min"},
    ])

    fig = go.Figure(data=[
        go.Bar(name="Current", x=["TPM", "Quota"], y=[old["tokens_per_minute"], old["token_quota"]],
               marker_color="#0078d4", opacity=0.85),
        go.Bar(name="New",     x=["TPM", "Quota"], y=[tpm_in.value, quota_in.value],
               marker_color="#107c10", opacity=0.85),
    ])
    fig.update_layout(
        barmode="group",
        title=f"Budget Change \u2014 {old['display_name']}",
        height=280, margin=dict(l=50, r=20, t=45, b=35),
        legend=dict(orientation="h", y=-0.25),
    )

    with apply_out:
        clear_output(wait=True)
        display(HTML("<b>Comparison</b>"))
        display(
            comparison.style
            .set_table_styles([
                {"selector": "thead th", "props": [("background-color","#107c10"),("color","white"),("font-weight","bold"),("padding","7px 12px")]},
                {"selector": "td",       "props": [("padding","5px 12px")]},
            ])
            .hide(axis="index")
        )
        fig.show()

    apply_btn.disabled = False

def on_apply(b):
    apply_btn.disabled = True
    slug = slug_dd.value
    with apply_out:
        print(f"Patching terraform.tfvars for '{slug}'...")
    patch_tfvars(slug, tpm_in.value, quota_in.value, period_dd.value)
    with apply_out:
        print("Running terraform apply (~30 s)...")
    result = subprocess.run(
        ["terraform", "apply", "-auto-approve"],
        capture_output=True, text=True, cwd=str(PROJECT_DIR), timeout=300,
    )
    if result.returncode == 0:
        global tenant_cfg, tenant_keys
        tenant_cfg  = tf_output("tenant_config")
        tenant_keys = tf_output("apim_tenant_subscription_keys")
        with apply_out:
            print("Done! Re-run cell 4 to refresh the overview table and charts.")
    else:
        with apply_out:
            print(f"ERROR:\n{result.stderr[-500:]}")
    apply_btn.disabled = False

preview_btn.on_click(on_preview)
apply_btn.on_click(on_apply)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:4px 0 10px'>Configure Tenant Quota</h3>"),
    slug_dd,
    widgets.HBox([tpm_in, quota_in, period_dd], layout=widgets.Layout(gap="12px")),
    widgets.HBox([preview_btn, apply_btn],       layout=widgets.Layout(gap="10px", margin="8px 0 0")),
    apply_out,
]))


## Quota Compliance Test

Send a configurable number of real requests through the APIM gateway and observe how the Redis-backed quota counter tracks spend.

The test produces:
- **Table** — per-request HTTP status, tokens used, and remaining quota/TPM from response headers
- **Bar chart** — tokens consumed per request (blue = success, red = blocked)
- **Burn-down line** — cumulative token spend vs the quota hard limit
- **Budget gauge** — percentage of the quota consumed in this test run


In [11]:
TEST_PROMPT = "In two sentences, describe what Azure API Management does and why it matters."

_t_opts   = [(f"{v['display_name']}  ({k})", k) for k, v in tenant_cfg.items()]

t_dd      = widgets.Dropdown(options=_t_opts, description="Tenant:",
                             layout=widgets.Layout(width="360px"))
n_req_w   = widgets.IntSlider(min=1, max=25, value=8, description="Requests:",
                              continuous_update=False, layout=widgets.Layout(width="440px"))
max_tok_w = widgets.IntSlider(min=50, max=2000, value=800, step=50, description="Max tokens:",
                              continuous_update=False, layout=widgets.Layout(width="440px"))
run_btn   = widgets.Button(description=" Run Test", button_style="primary", icon="play",
                           layout=widgets.Layout(width="150px"))
test_out  = widgets.Output()

def on_run(b):
    run_btn.disabled = True
    slug    = t_dd.value
    cfg     = tenant_cfg[slug]
    api_key = tenant_keys[slug]["primary_key"]
    quota   = cfg["token_quota"]
    period  = cfg["token_quota_period"]

    results, cumulative = [], 0

    with test_out:
        clear_output(wait=True)
        print(f"Sending {n_req_w.value} request(s) to {cfg['display_name']}  "
              f"(quota={quota:,} tokens / {period},  TPM cap={cfg['tokens_per_minute']:,})\n")

    for i in range(n_req_w.value):
        status, body, headers = chat(gateway, api_key, TEST_PROMPT, max_tokens=max_tok_w.value)
        ts = datetime.now(timezone.utc).strftime("%H:%M:%S")
        rq = headers.get("x-remaining-quota-tokens")
        rt = headers.get("x-remaining-tpm-tokens")

        if status == 200:
            used        = body.get("total_tokens", 0)
            cumulative += used
            results.append({
                "Req": i + 1, "UTC": ts, "HTTP": status,
                "Tokens": used, "Cumulative": cumulative,
                "Remaining Quota": int(rq) if rq is not None else None,
                "Remaining TPM":   int(rt) if rt is not None else None,
                "Result": "OK",
            })
        else:
            msg = ""
            if isinstance(body, dict):
                msg = body.get("message") or body.get("error", {}).get("message", "")
            label = ("QUOTA EXHAUSTED" if "quota" in msg.lower()
                     else "BLOCKED" if status in (403, 429) else f"ERR-{status}")
            results.append({
                "Req": i + 1, "UTC": ts, "HTTP": status,
                "Tokens": 0, "Cumulative": cumulative,
                "Remaining Quota": 0,
                "Remaining TPM":   int(rt) if rt is not None else 0,
                "Result": label,
            })
            break

    run_btn.disabled = False

    df      = pd.DataFrame(results)
    used_ok = df[df["Result"] == "OK"]
    blocked = df[df["Result"] != "OK"]
    total   = int(df["Tokens"].sum())
    pct     = round(min(total / quota * 100, 100), 1) if quota else 0

    def _col(val):
        if val == "OK":             return "color: #107c10; font-weight:bold"
        if "EXHAUSTED" in str(val): return "color: #d13438; font-weight:bold"
        return "color: #ca5010; font-weight:bold"

    with test_out:
        clear_output(wait=True)

        display(HTML(f"<h4>Results \u2014 {cfg['display_name']}</h4>"))
        try:   styled = df.style.map(_col, subset=["Result"])
        except AttributeError: styled = df.style.applymap(_col, subset=["Result"])
        display(
            styled
            .format({
                "Tokens":          "{:,}",
                "Cumulative":      "{:,}",
                "Remaining Quota": lambda v: f"{v:,}" if v is not None else "\u2014",
                "Remaining TPM":   lambda v: f"{v:,}" if v is not None else "\u2014",
            })
            .set_table_styles([
                {"selector": "thead th",                "props": [("background-color","#0078d4"),("color","white"),("font-weight","bold"),("padding","8px 13px")]},
                {"selector": "td",                      "props": [("padding","6px 13px")]},
                {"selector": "tr:nth-of-type(even) td", "props": [("background-color","#f0f7ff")]},
            ])
            .hide(axis="index")
        )

        # Chart 1: tokens per request
        fig_bar = go.Figure()
        if not used_ok.empty:
            fig_bar.add_trace(go.Bar(x=used_ok["Req"], y=used_ok["Tokens"],
                                     marker_color="#0078d4", name="Tokens used"))
        if not blocked.empty:
            fig_bar.add_trace(go.Bar(x=blocked["Req"], y=[max_tok_w.value] * len(blocked),
                                     marker_color="#d13438", name=str(blocked.iloc[0]["Result"])))
        fig_bar.update_layout(
            title=f"{cfg['display_name']} \u2014 Tokens per Request",
            xaxis=dict(title="Request #", dtick=1), yaxis_title="Tokens",
            height=310, margin=dict(l=50, r=20, t=48, b=35),
            legend=dict(orientation="h", y=-0.3),
        )
        fig_bar.show()

        # Chart 2: cumulative burn-down
        if not used_ok.empty:
            fig_line = go.Figure()
            fig_line.add_trace(go.Scatter(
                x=used_ok["Req"], y=used_ok["Cumulative"],
                mode="lines+markers", fill="tozeroy",
                line=dict(color="#0078d4", width=2), name="Consumed",
            ))
            fig_line.add_hline(
                y=quota, line_dash="dash", line_color="#d13438",
                annotation_text=f"Quota limit  ({quota:,})",
                annotation_position="top right",
            )
            if not blocked.empty:
                fig_line.add_vline(
                    x=int(blocked.iloc[0]["Req"]), line_dash="dot", line_color="#d13438",
                    annotation_text="Blocked", annotation_position="top left",
                )
            fig_line.update_layout(
                title=f"{cfg['display_name']} \u2014 Cumulative Quota Burn",
                xaxis=dict(title="Request #", dtick=1), yaxis_title="Cumulative tokens",
                height=330, margin=dict(l=50, r=20, t=48, b=35),
            )
            fig_line.show()

        # Chart 3: budget gauge
        n_blocked = len(blocked)
        note = f"{n_blocked} request(s) blocked" if n_blocked else "no requests blocked"
        fig_gauge = go.Figure(go.Indicator(
            mode="gauge+number+delta",
            value=pct,
            number={"suffix": "%", "font": {"size": 48}},
            delta={"reference": 100, "decreasing": {"color": "#107c10"}, "increasing": {"color": "#d13438"}},
            title={"text": (
                f"Budget Consumed \u2014 {cfg['display_name']}<br>"
                f"<span style='font-size:13px'>{total:,} / {quota:,} tokens \u00b7 {note}</span>"
            )},
            gauge={
                "axis":  {"range": [0, 100]},
                "bar":   {"color": "#0078d4"},
                "steps": [
                    {"range": [0, 70],   "color": "#c8f0c0"},
                    {"range": [70, 90],  "color": "#fff4ce"},
                    {"range": [90, 100], "color": "#fde7e9"},
                ],
                "threshold": {"line": {"color": "#d13438", "width": 4}, "thickness": 0.75, "value": 100},
            },
        ))
        fig_gauge.update_layout(height=380, margin=dict(l=20, r=20, t=70, b=20))
        fig_gauge.show()

run_btn.on_click(on_run)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:4px 0 10px'>Quota Compliance Test</h3>"),
    t_dd,
    n_req_w,
    max_tok_w,
    run_btn,
    test_out,
]))
